# Study 803 — Realized-Skewness Reversal 🎲📉

**Do stocks with a lottery-like recent tape go on to earn *less*?**

Amaya, Christoffersen, Jacobs & Vasquez (2015) find that a stock's **realized
skewness** predicts its cross-section of returns *negatively*: the names whose recent
return distribution is most **right-skewed** under-earn, and a long **low-skew** /
short **high-skew** book earns a positive spread. We take the self-contained daily
version on a liquid US cross-section (2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

A right-skewed name has a fat *upside* tail — the occasional big up-day. Skewness-loving investors overpay for that lottery ticket, so the name is priced too high and its *future* return is lower. Sort on the trailing third moment; buy the boring left-skewed names, sell the lottery tickets.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-3.19, t_nw=-3.04, lo_bps=6.15, hi_bps=9.34, gross_sharpe=-0.74)
print('long low-skew / short high-skew spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-skew book %+.2f bps vs high-skew book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long low-skew / short high-skew spread: -3.19 bps/day (NW t = -3.04)
  low-skew book +6.15 bps vs high-skew book +9.34 bps
  gross spread Sharpe (before cost): -0.74


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, skew present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from realized_skewness import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=803, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=803, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +0.56  (should be ~0)
planted world: spread NW t = +7.70  (should light up)


## 3. The honest verdict — the famous edge does *not* replicate here

On this liquid mega-cap tape the long-low-skew / short-high-skew spread is **-3.19 bps/day** with NW *t* = **-3.04** — significant, but with the **opposite sign** to Amaya et al: here the lottery-like high-skew names actually *out-earned* the boring low-skew ones (the permutation null centres at 0 with sd 0.87 bps; the observed value is ~3.6σ into the *left* tail). The seeded synthetic control recovers a *planted* Amaya relation cleanly, so this is a genuine sign-reversal on the mega-cap survivor universe, not a bug — the realized-skewness premium is a small-and-illiquid-stock phenomenon that does not survive on 50 mega-caps. **Signal: None** (the claimed edge is absent), **Tradability: Mirage**.